# 03 · Speaker diarization (NVIDIA Streaming Sortformer)

Runs in the `egyspeech-nemo` env. Up to 4 speakers per episode; the frame probabilities are kept so the segmenter can exclude every overlap.

In [ ]:
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO)); os.chdir(REPO)
if Path("/workspace").exists():
    os.environ.setdefault("HF_HOME", "/workspace/hf_cache")
import numpy as np, pandas as pd
import plotly.express as px
from egyspeech import nb
cfg, lay = nb.cfg_and_layout()          # configs/config.yaml
print("work dir:", lay.root)

In [ ]:
nb.run("diarize")

In [ ]:
vid = next(p.stem for p in sorted((lay.root / "diar").glob("*.json")))
info = json.loads(lay.diar_json(vid).read_text())
probs = np.load(lay.diar_probs(vid)).astype(float)
print(vid, "speakers:", info["n_speakers"], "| duration", round(info["duration"] / 60, 1), "min")
t = np.arange(len(probs)) * info["frame_sec"]
view = t < 300
fig = px.line(pd.DataFrame(probs[view], columns=[f"spk{i}" for i in range(probs.shape[1])]).assign(t=t[view]), x="t", y=[f"spk{i}" for i in range(probs.shape[1])], title="Speaker activity (first 5 min)")
fig.show()
seg = pd.DataFrame(info["segments"], columns=["start", "end", "speaker"])
display(seg.assign(dur=seg.end - seg.start).groupby("speaker")["dur"].sum().div(60).round(1).rename("minutes"))